# Auto-PPT Agent — Assignment 1 (MCP + Claude)

## Objective

Turn a **single-sentence** user prompt into a working **`.pptx`** file using an **agentic loop** (plan first, then tool calls), with **MCP** servers for PowerPoint and research.

## Deliverables in this repo (separate files)

| File | Purpose |
|------|--------|
| `ppt_mcp_server.py` | MCP tools: create presentation, add slides, save file |
| `research_mcp_server.py` | MCP tool: lightweight topic lookup |
| `agent_ppt.py` | Agent: planning + calls both MCP servers + optional Claude |

This notebook **drives** `agent_ppt.py` so you can show outputs for grading.

## AI assistance disclosure (required by course policy)

Notebook structure, markdown cells, import-line comments, and some error-handling wording were drafted or refined with assistance from **Cursor** (AI-assisted editing). I reviewed all changes for correctness against the assignment, understood the MCP + agent flow, and executed the notebook and related scripts to verify behavior.

## Agentic loop (assignment requirement)

1. **Plan** slide titles (global outline) before writing any slide content.
2. **Execute** per slide: research (optional) → bullets → MCP `add_slide`.
3. **Save** a valid `.pptx` via MCP `save_presentation`.

This is **not** a single LLM call; it mirrors the loop described in `Agentic_assignment_1.md`.

## Setup (run once per machine)

From the `ASSIGNMENT` folder in PowerShell:

```powershell
python -m venv .venv
.\.venv\Scripts\activate
pip install -r requirements.txt
```

Optional — Claude API (better outlines/bullets; agent still runs without it):

```powershell
setx ANTHROPIC_API_KEY "your_key_here"
```

Restart the terminal after `setx`.

**Jupyter:** select the kernel that uses the same venv (`python -m ipykernel install` if needed).

## Identifier glossary (short)

| Name | Meaning |
|------|--------|
| `NOTEBOOK_DIR` | Folder containing `agent_ppt.py` (resolved from `__file__`, cwd, or a direct subfolder) |
| `DemoRunConfig` | Demo prompt plus `OUTPUT_PATH` / `SOURCE_PPTX` (set by your answers in the config cell) |
| `run_demo_async` | Async function that calls `run_ppt_agent` with error handling |
| `result_path` | String path returned after the `.pptx` is saved |

---
### Cell 1 — Imports

Standard library and typing only here so failures are easy to read.

In [ ]:
import asyncio  # Needed because run_ppt_agent is async (MCP stdio clients).
import os  # Needed to read ANTHROPIC_API_KEY for optional Claude calls.
import sys  # Needed to add ASSIGNMENT folder to import path when kernel cwd differs.
import traceback  # Needed to print full stack traces only when we choose (debugging).
from pathlib import Path  # Needed for cross-platform paths (Windows/Linux).

### Cell 2 — Put `ASSIGNMENT` on `sys.path`

**Why:** Jupyter’s current working directory might be Cursor’s workspace root instead of this folder. `_resolve_assignment_dir()` finds the directory that contains `agent_ppt.py` (walks up from `cwd` and checks direct subfolders), then sets `NOTEBOOK_DIR` so imports and `output\…` paths match your real `ASSIGNMENT` tree.

In [ ]:
def _resolve_assignment_dir() -> Path:
    """Folder that contains agent_ppt.py — works even when Jupyter cwd is Cursor's workspace root."""
    try:
        here = Path(__file__).resolve().parent
        if (here / "agent_ppt.py").is_file():
            return here
    except NameError:
        pass
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "agent_ppt.py").is_file():
            return p
    for sub in start.iterdir():
        if sub.is_dir() and (sub / "agent_ppt.py").is_file():
            return sub.resolve()
    return start


NOTEBOOK_DIR = _resolve_assignment_dir()

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))  # Prepend so `import agent_ppt` finds local module first.

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)  # Should end with ...\ASSIGNMENT when this folder is on disk nearby.

### Cell 3 — Import the agent module

**Why try/except:** If the kernel cwd is wrong, `agent_ppt` is missing and we fail with a **clear** message instead of a vague `ModuleNotFoundError`.

In [ ]:
try:
    from agent_ppt import run_ppt_agent  # Main entry: connects MCP servers and writes pptx.
except ImportError as exc:
    # Re-raise with actionable guidance because grading time is limited.
    raise ImportError(
        "Could not import agent_ppt. Set kernel cwd to ASSIGNMENT or open Jupyter from that folder."
    ) from exc

### Cell 4 — Config + **your save / edit paths**

**Why a class:** satisfies “classes/objects” rubric. When you run the next cell, it **asks** where to save the `.pptx` and optionally which existing deck to **open** so new slides are **appended** (then saved to your chosen path).

In [ ]:
class DemoRunConfig:
    """Holds the demo prompt; paths are filled in below when you run this cell."""

    # Single-sentence prompt as required by Agentic_assignment_1.md.
    PROMPT = (
        "Create a 5-slide presentation on the life cycle of a star for a 6th-grade class"
    )


def _notebook_user_path(raw: str, default_rel: str) -> Path:
    """Turn user input into an absolute Path (relative paths are under NOTEBOOK_DIR)."""
    text = (raw or default_rel).strip().strip('"')
    p = Path(text)
    if not p.is_absolute():
        p = (NOTEBOOK_DIR / p).resolve()
    else:
        p = p.expanduser().resolve()
    return p


_save_in = input(
    "Path to save new .pptx (relative to ASSIGNMENT or absolute) [output/notebook_auto_ppt_output.pptx]: "
).strip()
DemoRunConfig.OUTPUT_PATH = _notebook_user_path(
    _save_in, "output/notebook_auto_ppt_output.pptx"
)

_edit_in = input(
    "Optional: existing .pptx to open and append new slides to (Enter = brand-new deck): "
).strip()
DemoRunConfig.SOURCE_PPTX = None
if _edit_in:
    ep = _notebook_user_path(_edit_in, _edit_in)
    if ep.is_file():
        DemoRunConfig.SOURCE_PPTX = str(ep)
    else:
        print("Warning: file not found:", ep, "— using a new deck.")

print("Will save to:", DemoRunConfig.OUTPUT_PATH)
print("Load & append from:", DemoRunConfig.SOURCE_PPTX or "(new deck)")

### Cell 5 — Environment check (non-fatal)

**Why:** Shows whether Claude will run; agent still produces a file using fallbacks if the key is absent.

In [ ]:
has_claude_key = bool(os.getenv("ANTHROPIC_API_KEY", "").strip())  # True if non-empty key.
print("ANTHROPIC_API_KEY set:", has_claude_key)  # Grader-visible boolean.
if not has_claude_key:
    print("Note: running without Claude — outlines/bullets use built-in fallbacks.")  # Sets expectations.

### Cell 6 — Async runner with exceptions

**Why:** `run_ppt_agent` is async; we wrap it so one cell can `await` via `asyncio.run` in the next cell. Inner `try/except` maps failures to readable messages.

In [ ]:
async def run_demo_async() -> str:
    """Call the MCP-backed agent and return the saved .pptx path as a string."""

    out_path = DemoRunConfig.OUTPUT_PATH  # Absolute path from your input in the config cell.
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)  # Create parent folders if missing.
    except OSError as exc:
        # Disk permission or invalid path should not look like an MCP bug.
        raise OSError(f"Cannot create output directory: {out_path.parent}") from exc

    try:
        # Optional SOURCE_PPTX loads an existing deck; new slides are appended, then save goes to out_path.
        saved = await run_ppt_agent(
            DemoRunConfig.PROMPT, str(out_path), DemoRunConfig.SOURCE_PPTX
        )
        return saved  # Return path string for display and assertions.
    except RuntimeError as exc:
        # Raised inside agent when MCP create/open presentation returns unexpected payload.
        raise RuntimeError("Agent failed during MCP tool calls (check server stderr).") from exc
    except Exception as exc:
        # Catch-all with context so graders see it was during demo run.
        raise type(exc)(f"run_ppt_agent failed: {exc}") from exc

### Cell 7 — Execute (top-level try / except)

**Run this cell last.** Prints the result path or a controlled error message.

In [ ]:
result_path = None  # Will hold returned path string on success.

try:
    result_path = asyncio.run(run_demo_async())  # Blocks until async agent finishes.
except KeyboardInterrupt:
    print("Stopped by user (Kernel interrupt).")  # Friendly message if user stops cell.
except Exception as exc:
    print("ERROR:", type(exc).__name__, "—", exc)  # Short message for notebook readability.
    traceback.print_exc()  # Full traceback for debugging (still visible in notebook output).
else:
    print("Presentation created:", result_path)  # Success line for demos / grading.

result_path  # Last expression shows path in notebook output (if no exception).

### Cell 8 — Optional: verify file exists

**Why:** Proves correctness (40% rubric) — file on disk, not just a printed string.

In [ ]:
check_path = DemoRunConfig.OUTPUT_PATH  # Same absolute path as in run_demo_async.

try:
    exists = check_path.is_file()  # True only if a real file was written.
    size_bytes = check_path.stat().st_size if exists else 0  # Non-zero size suggests valid pptx.
    print("File exists:", exists, "| size (bytes):", size_bytes)  # Grader-visible proof.
except OSError as exc:
    print("Could not stat output file:", exc)  # Handle rare permission issues cleanly.

## CLI alternative (same agent code)

Interactive terminal: you are **prompted** for the save path (Enter = `output\auto_ppt_output.pptx`) and optionally an existing `.pptx` to append to.

```powershell
python .\agent_ppt.py "Create a 5-slide presentation on the life cycle of a star for a 6th-grade class"
```

Non-interactive / scripts: pass paths explicitly.

```powershell
python .\agent_ppt.py "Your one-line prompt" --output "output\my_deck.pptx" --no-prompt
python .\agent_ppt.py "Add 3 slides about Mars" --edit "output\old_deck.pptx" --output "output\updated.pptx" --no-prompt
```

## Rubric checklist (self-check)

- **MCP:** 2 servers (PPT + research) — see `ppt_mcp_server.py`, `research_mcp_server.py`.
- **Agentic loop:** plan before slides — see `agent_ppt.py`.
- **Output:** `.pptx` on disk — this notebook verifies existence in Cell 8.
- **Robustness:** fallbacks if Claude/search fails — implemented in `agent_ppt.py` / research server.
- **This notebook:** markdown structure, comments, exceptions, modular `DemoRunConfig` + `run_demo_async`.

## Reflection (`Agentic_assignment_1.md` deliverable)

1. Where did your agent fail its first attempt?
   - 

2. How did MCP prevent you from writing hardcoded scripts?
   - 

Also fill `reflection.md` in the same folder if your instructor wants a separate file.